# Broken access control: owner-scoped history (2.7)

Once accounts exist, one user must not be able to read or delete another user's saved lookups. 
Every history query is scoped to the signed-in user's id, so asking for someone else's record id just returns 404. 
This is an insecure direct object reference (IDOR), and demonstrates user B fetching user A's record being refused.

To show the attack succeeding without the defence as well, this notebook flips an
`ACCESS_CONTROL` switch. With it off, the owner filter is dropped and any signed-in user
can read or delete any record by id. With it on, the same request is a 404 (as to not reveal what records exist)

In [ ]:
import sys
import uuid
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
BACKEND = HERE if (HERE / "app" / "main.py").is_file() else HERE.parent
SEC = BACKEND / "security_eval"
sys.path.insert(0, str(BACKEND))

import harness as h

h.ensure_db()
PW = "not-a-real-password"

 Container model-backend-database-1 Running 


admin already exists! Skipping...
user already exists! Skipping...
joey already exists! Skipping...
sienna already exists! Skipping...
luke already exists! Skipping...


## The attack: user B reaches user A's record by id

In [ ]:
def run_idor(access_control):
    proc = h.start_server(ACCESS_CONTROL=access_control)
    try:
        tag = uuid.uuid4().hex
        victim, attacker = f"victim_{tag}", f"attacker_{tag}"           # random username and password for repeatability

        # create accts
        h.post("/api/register", {"username": victim, "password": PW})
        h.post("/api/register", {"username": attacker, "password": PW})
        
        # login toks
        vtok = h.login(victim, PW)
        atok = h.login(attacker, PW)

        # the victim saves a check, giving us a record id that belongs to them
        h.post("/api/check", {"url": "paypal-account-verify.top/login"}, token=vtok)
        target_id = h.get("/api/history", token=vtok)[1][0]["id"]

        # the attacker, signed in as themselves, directly tries reading the victim's record id
        read_status, read_body, _ = h.get(f"/api/history/{target_id}", token=atok)
        del_status, _, _          = h.delete(f"/api/history/{target_id}", token=atok)
    finally:
        h.stop_server(proc)

    leaked = read_status == 200 or del_status == 204
    return {
        "access_control": "on" if access_control else "off",
        "read_status": read_status,
        "delete_status": del_status,
        "leaked_url": read_body.get("url") if isinstance(read_body, dict) else None,
        "outcome": "LEAKED - reached another user's record" if leaked else "BLOCKED - 404, stays private",
    }


off = run_idor(access_control=False)
on  = run_idor(access_control=True)
print(off)
print(on)

done


## Results

In [3]:
table = pd.DataFrame([off, on])[["access_control", "read_status", "delete_status", "outcome"]]
print(table.to_string(index=False))
table.to_csv(SEC / "access_control_results.csv", index=False)

print()
print("defence OFF: the attacker read the victim's saved URL:", off["leaked_url"])
print(f"defence ON : read {on['read_status']}, delete {on['delete_status']} - the victim's record is untouchable")

access_control  read_status  delete_status                                outcome
           off          200            204 LEAKED - reached another user's record
            on          404            404           BLOCKED - 404, stays private

defence OFF: the attacker read the victim's saved URL: paypal-account-verify.top/login
defence ON : read 404, delete 404 - the victim's record is untouchable


## What the numbers say

With `ACCESS_CONTROL` off, the attacker read the victim's saved URL and deleted the record
with a plain GET and DELETE, using only the record's id. With it on, both requests are 404,
because every query also filters on the signed-in user's id. Returning 404 rather than 403
is deliberate: it does not even confirm the record exists.

What the defence does not fix:

- It guards the history records. Any new endpoint that takes an id has to apply the same
  owner filter, so the defence is a habit to repeat on all routes
- Ids are sequential integers and easy to guess. The owner check is what makes that
  harmless; hiding the ids would not be enough on its own. UUID is hence better

Cost: effectively none. Each query carries the user ID it already has from the token, so
there is no extra round trip and nothing a legitimate user would notice.